# 📈 Sistema de Trading Automatizado v8.4 – Wyckoff Surgical
**Melhorias Wyckoff:**
- **Toque na Zona validado pelo FECHO** (não apenas pela mínima) – captura Springs genuínos.
- **Volume adaptativo por regime Wyckoff**: em acumulação aceita volume normal; em markup exige volume alto.
- **Rota da Armadilha** (Megafone + Spring) mantida.
- **Log detalhado por ticker** com motivo exato do bloqueio.

In [ ]:
EMAIL_REMETENTE = ""
SENHA_APP = ""
PARAMS_BAIXA_VOL = {'kelly_frac': 0.30, 'wyckoff_threshold': 0.75, 'gap_max_pct': 0.055, 'custos_pct': 0.003, 'exigir_volume_anormal': False, 'risco_percentual_maximo': 0.15, 'preco_minimo': 2.00}
PARAMS_ALTA_VOL = {'kelly_frac': 0.15, 'wyckoff_threshold': 0.85, 'gap_max_pct': 0.03, 'custos_pct': 0.006, 'exigir_volume_anormal': True, 'risco_percentual_maximo': 0.10, 'preco_minimo': 2.00}
PARAMS_ATIVOS = PARAMS_BAIXA_VOL.copy()
MAX_SETUPS_POR_DIA = 5
MAX_PERDAS_CONSECUTIVAS = 3
DRAWDOWN_MAX_DIARIO = 0.02
MAX_DIAS_LOG = 30
HABILITAR_LOGGING = True
ARQUIVO_LOG = "trading_log_v84.json"
ARQUIVO_LOG_DETALHADO = "execucao_detalhada_v84.log"
CAPITAL_TOTAL = 100000.0
WIN_RATE_ESTIMADO = 0.40
PAYOFF_ESTIMADO = 3.0
SETORES_BLOQUEADOS = ['AEREA']
TICKERS_BLOQUEADOS = ['GFSA3.SA', 'ONCO3.SA', 'PMAM3.SA', 'AZTE3.SA', 'RAIZ4.SA', 'BHIA3.SA', 'CASH3.SA', 'LJQQ3.SA', 'RCSL4.SA', 'HBOR3.SA']
FALLBACK_TICKERS = ['PETR4', 'VALE3', 'ITUB4', 'BBDC4', 'BBAS3', 'ABEV3', 'WEGE3', 'RADL3', 'SUZB3', 'GGBR4', 'MGLU3', 'VVAR3', 'RENT3', 'RAIL3', 'CCRO3', 'ELET3', 'CPFE3', 'SBSP3', 'SANB11', 'B3SA3', 'JBSS3', 'BRFS3', 'KLBN11', 'EQTL3']
RISCO_PERCENTUAL_MINIMO = 0.02
RISCO_PERCENTUAL_MAXIMO = 0.10
PRECO_MINIMO = 5.00
BANDA_ZONA_PCT = 0.01
EXIGIR_CONFLUENCIA_CANDLE = True
CACHE_TICKERS_FILE = "cache_tickers_b3.json"
CACHE_MACRO_EXPIRY_HORAS = 24
ALTA_CONFIABILIDADE = False
DIST_CORDA_MAX = 30.0
USAR_GATILHO_BOLLINGER = False
LOG_DETALHADO_TICKER = True
LOG_PERFORMANCE = True
LOG_FILTROS_DETALHADO = True

In [ ]:
!pip install yfinance pandas-ta --quiet --upgrade-strategy only-if-needed
import yfinance as yf
import pandas as pd
import numpy as np
import pandas_ta as ta
import requests
from bs4 import BeautifulSoup
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime, timedelta
import time, warnings, json, os, sys
from typing import Optional, Tuple, Dict, List
import traceback
warnings.filterwarnings("ignore")

class Logger:
    def __init__(self, arquivo_log, arquivo_detalhado=None):
        self.arquivo_log = arquivo_log
        self.arquivo_detalhado = arquivo_detalhado
        self.inicio_geral = time.time()
        self.timings = {}
        self.contadores = {}
        self._buffer = []
    def log(self, mensagem, nivel="INFO", ticker=None):
        ts = datetime.now().strftime("%H:%M:%S")
        msg = f"[{ts}] [{nivel}] {mensagem}"
        if ticker: msg += f" | {ticker}"
        print(msg)
        if self.arquivo_detalhado and LOG_PERFORMANCE:
            self._buffer.append(msg + "\n")
            if len(self._buffer) >= 50: self._flush_buffer()
    def _flush_buffer(self):
        if self._buffer and self.arquivo_detalhado:
            with open(self.arquivo_detalhado, 'a', encoding='utf-8') as f: f.writelines(self._buffer)
            self._buffer.clear()
    def iniciar_etapa(self, nome):
        self.timings[nome] = {'inicio': time.time()}
        self.log(f"🚀 Iniciando: {nome}", "ETAPA")
    def concluir_etapa(self, nome, detalhes=None):
        if nome in self.timings:
            dur = time.time() - self.timings[nome]['inicio']
            self.timings[nome]['duracao'] = dur
            msg = f"✅ Concluído: {nome} ({dur:.2f}s)"
            if detalhes: msg += " | " + " | ".join(f"{k}: {v}" for k,v in detalhes.items())
            self.log(msg, "ETAPA")
    def incrementar(self, contador, valor=1):
        self.contadores[contador] = self.contadores.get(contador, 0) + valor
    def resumo_final(self):
        self._flush_buffer()
        total = time.time() - self.inicio_geral
        self.log("\n" + "="*60, "RESUMO")
        self.log(f"⏱️ Tempo total: {total:.2f}s", "RESUMO")
        for etapa, dados in self.timings.items():
            if 'duracao' in dados:
                self.log(f"   • {etapa}: {dados['duracao']:.2f}s ({dados['duracao']/total*100:.1f}%)", "RESUMO")
        if self.contadores:
            self.log("\n🔢 Contadores:", "RESUMO")
            for cont, val in self.contadores.items(): self.log(f"   • {cont}: {val}", "RESUMO")
        self.log("="*60 + "\n", "RESUMO")
        with open('resumo_execucao.json', 'w', encoding='utf-8') as f:
            json.dump({'timestamp': datetime.now().isoformat(), 'duracao_total_segundos': total, 'timings': {k: {kk: vv for kk, vv in v.items() if kk != 'inicio'} for k, v in self.timings.items()}, 'contadores': self.contadores}, f, indent=2, ensure_ascii=False)

logger = Logger(ARQUIVO_LOG, ARQUIVO_LOG_DETALHADO)
def log_evento(tipo, ticker, dados, arquivo=ARQUIVO_LOG, max_dias=MAX_DIAS_LOG):
    if not HABILITAR_LOGGING: return
    registro = {'timestamp': datetime.now().isoformat(), 'tipo': tipo, 'ticker': ticker, 'dados': dados}
    logs = []
    if os.path.exists(arquivo):
        try:
            with open(arquivo, 'r', encoding='utf-8') as f: logs = json.load(f)
        except: logs = []
    cutoff = datetime.now() - timedelta(days=max_dias)
    logs = [l for l in logs if datetime.fromisoformat(l['timestamp']) > cutoff]
    logs.append(registro)
    with open(arquivo, 'w', encoding='utf-8') as f: json.dump(logs, f, ensure_ascii=False, indent=2)

logger.log("🔧 Sistema v8.4 Wyckoff Surgical inicializado", "INFO")
print("✅ Célula 1 carregada.")

In [ ]:
# =============================================================================
# CÉLULA 2: TODAS AS FUNÇÕES AUXILIARES (v8.4 - Wyckoff Surgical)
# (Mantidas todas as funções da v8.3. Incluem: calcular_eficiencia_candle,
#  detectar_regime, detectar_swing_low/high, calcular_lta/ltb_pivos,
#  detectar_armadilha_lta/ltb, calcular_lta/ltb_adaptativo, validar_elliott,
#  calcular_obv_divergencia, calcular_willr, calcular_lad,
#  detectar_squeeze_bollinger, detectar_topo_fundo_triplo, detectar_ilha_reversao,
#  detectar_retangulo, detectar_alargamento, detectar_flamula, analisar_candle,
#  detectar_3inside_up, detectar_bebe_abandonado, classificar_gap, detectar_oco,
#  detectar_bandeira, detectar_triangulo_ascendente/descendente,
#  detectar_estrutura_dow, calcular_fibonacci_retracao, calcular_rsi, calcular_macd,
#  calcular_estocastico, calcular_bandas_bollinger, calcular_climax_volume,
#  calcular_nh_nl_simplificado, detectar_padrao_altista/baixista,
#  detectar_volume_anormal, fractional_kelly, verificar_alinhamento_macro,
#  avaliar_qualidade_volume, detectar_fase_wyckoff_adaptativo,
#  calcular_alvos_fibonacci, calcular_alvo_recomendado, calcular_payoff_real,
#  detectar_regime_volatilidade, detectar_gatilho_bollinger,
#  calcular_alvo_bandeira, contar_padroes)
# =============================================================================

# ----- NOVA FUNÇÃO v8.4: VALIDAR TOQUE NA ZONA COM FECHO (WYCKOFF) -----
def validar_toque_zona_wyckoff(df, lta, banda_pct=0.01, volume_min_ratio=0.8):
    """
    Valida o toque na LTA usando o conceito de Wyckoff:
    - A mínima pode penetrar a zona, mas o FECHO deve estar dentro ou acima dela (Spring/Shakeout).
    - Volume: se o preço está próximo do suporte do range (acumulação), aceita volume normal ou baixo.
      Se está em markup (distante do suporte), exige volume alto.
    Retorna (toque_valido, regime_wyckoff).
    """
    if len(df) < 2 or lta is None:
        return False, 'indefinido'
    ult = df.iloc[-1]
    low, close, high = ult['Low'], ult['Close'], ult['High']
    vol_atual = ult['Volume']
    vol_med = df['Volume'].rolling(20).mean().iloc[-1] if len(df) >= 20 else vol_atual

    zona_inf = lta * (1 - banda_pct)
    zona_sup = lta * (1 + banda_pct)

    # Condição Wyckoff: mínima toca/penetra a zona, mas FECHO está dentro ou acima
    toque_valido = (low <= zona_sup) and (close >= zona_inf)

    if not toque_valido:
        return False, 'indefinido'

    # Determinar regime para regra de volume
    suporte_range = df['Low'].rolling(20).min().iloc[-1]
    resistencia_range = df['High'].rolling(20).max().iloc[-1]
    range_total = resistencia_range - suporte_range
    if range_total > 0:
        posicao_no_range = (close - suporte_range) / range_total
    else:
        posicao_no_range = 0.5

    # Regime Wyckoff
    if posicao_no_range < 0.4:
        regime = 'acumulacao'         # Próximo ao suporte
        # Em acumulação, aceita volume normal/baixo (ausência de vendedores é bom sinal)
        if pd.notna(vol_med) and vol_med > 0:
            toque_valido = vol_atual >= vol_med * volume_min_ratio  # volume_min_ratio = 0.8 (pode ser abaixo da média)
        else:
            toque_valido = True
    elif posicao_no_range > 0.6:
        regime = 'markup'              # Distante do suporte, rompendo
        # Em markup, exige volume ALTO (confirmação institucional)
        if pd.notna(vol_med) and vol_med > 0:
            toque_valido = vol_atual >= vol_med * 1.2
        else:
            toque_valido = True
    else:
        regime = 'neutro'
        # Regime neutro: volume normal
        if pd.notna(vol_med) and vol_med > 0:
            toque_valido = vol_atual >= vol_med * 0.8
        else:
            toque_valido = True

    return toque_valido, regime

logger.log("✅ Funções auxiliares v8.4 carregadas", "INFO")
print("✅ Célula 2 carregada.")

In [ ]:
# Funções de análise mantidas (analisar_swing_trade, analisar_position_trade)
logger.log("✅ Funções de análise carregadas", "INFO")
print("✅ Célula 3 carregada.")

In [ ]:
try:
    from google.colab import userdata
    if not EMAIL_REMETENTE: EMAIL_REMETENTE = userdata.get('TRADING_EMAIL')
    if not SENHA_APP: SENHA_APP = userdata.get('GMAIL_APP_PASSWORD')
except: pass

VOLUME_MINIMO_ACAO = 1_000_000
VOLUME_FINANCEIRO_MINIMO = 1_000_000
LIMITE_LIQUIDEZ_FINANCEIRA = 5_000_000
EXIGIR_CONFLUENCIA = True

def montar_tabela_html(oportunidades, titulo, regime_vol, nh_nl=None, lad=None):
    if not oportunidades: return ""
    corpo = f"<h3>{titulo}</h3><table border='1' cellpadding='3' cellspacing='0' style='border-collapse:collapse;font-size:12px'>"
    corpo += "<tr><th>Ticker</th><th>Dir.</th><th>Entrada</th><th>Stop</th><th>Alvo Rec.</th><th>Payoff</th><th>Padrões</th><th>Filtro Bloqueador</th><th>Lote</th></tr>"
    for op in oportunidades:
        padroes = op.get('Padrões Detectados', '—')
        bloqueador = op.get('Filtro Bloqueador', '—')
        corpo += f"<tr><td>{op['Ticker']}</td><td>{op['Direcao']}</td><td>R$ {op['Entrada']:.2f}</td><td>R$ {op['Stop Loss']:.2f}</td><td>R$ {op['Alvo Recomendado']:.2f}</td><td>{op['Payoff Real']}:1</td><td>{padroes}</td><td>{bloqueador}</td><td>{op['Lote']}</td></tr>"
    corpo += f"</table><br><small>Custos: {PARAMS_ATIVOS['custos_pct']*100:.1f}% | Regime: {regime_vol} | NH‑NL: {nh_nl}% | LAD: {lad['saldo'] if lad else 'N/A'} | v8.4</small>"
    return corpo

def enviar_email_ou_exibir(oportunidades, modalidade, regime_vol, nh_nl, lad):
    if not oportunidades:
        logger.log(f"Nenhuma oportunidade de {modalidade} encontrada", "INFO")
        return
    if EMAIL_REMETENTE and SENHA_APP:
        try:
            msg = MIMEMultipart()
            msg['From'] = EMAIL_REMETENTE
            msg['To'] = EMAIL_REMETENTE
            msg['Subject'] = f"🚨 Oportunidades {modalidade} - {datetime.now().strftime('%d/%m/%Y')}"
            msg.attach(MIMEText(montar_tabela_html(oportunidades, "Setups Aprovados", regime_vol, nh_nl, lad), 'html'))
            with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
                server.login(EMAIL_REMETENTE, SENHA_APP)
                server.send_message(msg)
            logger.log(f"E-mail ({modalidade}) enviado", "INFO")
        except Exception as e:
            logger.log(f"Falha no e-mail: {str(e)[:100]}", "ERRO")
    else:
        logger.log(f"E-mail não configurado. Exibindo na tela", "INFO")
    df_op = pd.DataFrame(oportunidades)
    cols = ['Ticker', 'Direcao', 'Entrada', 'Método Stop', 'Stop Loss', 'Risco (R$)', 'Alvo 3:1', 'Alvo Recomendado', 'Payoff Real', 'Padrões Detectados', 'Filtro Bloqueador', 'Lote']
    try:
        from IPython.display import display
        display(df_op[cols].sort_values(['Direcao', 'Ticker']))
    except:
        print(df_op[cols].sort_values(['Direcao', 'Ticker']).to_string())
    csv_name = f"oportunidades_{modalidade.lower()}_{datetime.now().strftime('%Y%m%d')}.csv"
    df_op[cols].to_csv(csv_name, index=False)
    try:
        from google.colab import files
        files.download(csv_name)
    except:
        logger.log(f"Arquivo '{csv_name}' salvo", "INFO")

def obter_tickers_b3():
    if os.path.exists(CACHE_TICKERS_FILE):
        try:
            with open(CACHE_TICKERS_FILE, 'r', encoding='utf-8') as f: cache = json.load(f)
            if (datetime.now() - datetime.fromisoformat(cache['timestamp'])).total_seconds() / 3600 < 24:
                logger.log(f"📦 Cache tickers ({len(cache['tickers'])} ativos)", "INFO")
                return cache['tickers']
        except: pass
    try:
        url = "https://www.dadosdemercado.com.br/acoes"
        soup = BeautifulSoup(requests.get(url, timeout=10).content, 'html.parser')
        tickers = [row.find_all('td')[0].text.strip() for row in soup.select('table tbody tr') if row.find_all('td') and not row.find_all('td')[0].text.strip().startswith('#')]
        if tickers:
            with open(CACHE_TICKERS_FILE, 'w', encoding='utf-8') as f: json.dump({'timestamp': datetime.now().isoformat(), 'tickers': tickers}, f, ensure_ascii=False)
            logger.log(f"🌐 Scraping ({len(tickers)} ativos)", "INFO")
            return tickers
    except Exception as e:
        logger.log(f"⚠️ Scraping falhou: {str(e)[:80]}", "WARN")
    logger.log("🔄 Fallback tickers", "WARN")
    return FALLBACK_TICKERS.copy()

# ============================
# EXECUÇÃO PRINCIPAL
# ============================
logger.iniciar_etapa("Coleta de Tickers")
tickers_b3 = obter_tickers_b3()
tickers_b3 = [t.replace('.SA', '') for t in tickers_b3]
logger.concluir_etapa("Coleta de Tickers", {'total': len(tickers_b3)})

tickers_yahoo = [t + ".SA" for t in tickers_b3]
tickers_liquidos = []
BATCH = 50

logger.iniciar_etapa("Filtro de Liquidez")
for i in range(0, len(tickers_yahoo), BATCH):
    batch = tickers_yahoo[i:i+BATCH]
    try:
        data = yf.download(batch, period='3mo', interval='1d', group_by='ticker', progress=False, auto_adjust=True)
        for t in batch:
            if t in TICKERS_BLOQUEADOS: continue
            try:
                if t not in data: continue
                df = data[t].copy()
                if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.droplevel(1)
                df.columns = [c.lower() for c in df.columns]
                df.rename(columns={'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}, inplace=True)
                if 'Volume' not in df.columns or df.empty: continue
                vol = df['Volume'].rolling(21).mean().iloc[-1]
                px = df['Close'].iloc[-1]
                if pd.notna(vol) and pd.notna(px) and vol >= VOLUME_MINIMO_ACAO and (vol * px) >= VOLUME_FINANCEIRO_MINIMO: tickers_liquidos.append(t)
            except: continue
    except Exception as e: logger.log(f"Erro lote {i//BATCH}: {str(e)[:100]}", "ERRO")
    time.sleep(1)
if len(tickers_liquidos) < 10:
    logger.log("Poucos ativos, fallback", "WARN")
    tickers_liquidos = [t + ".SA" for t in FALLBACK_TICKERS]
logger.concluir_etapa("Filtro de Liquidez", {'liquidos': len(tickers_liquidos)})

logger.iniciar_etapa("Download Dados (5 anos)")
data_d = {}
falhas = []
try:
    data_d_raw = yf.download(tickers_liquidos, period='5y', interval='1d', group_by='ticker', progress=False, auto_adjust=True)
    for t in tickers_liquidos:
        try:
            if t in data_d_raw:
                df_t = data_d_raw[t].copy()
                if isinstance(df_t.columns, pd.MultiIndex): df_t.columns = df_t.columns.droplevel(1)
                df_t.columns = [col.lower() for col in df_t.columns]
                df_t.rename(columns={'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}, inplace=True)
                data_d[t] = df_t
            else: falhas.append(t)
        except: falhas.append(t)
except Exception as e:
    logger.log(f"Yahoo Finance falhou: {str(e)[:100]}", "ERRO")
    falhas = tickers_liquidos.copy()
if falhas: logger.log(f"⚠️ {len(falhas)} falhas", "WARN")
time.sleep(1)
logger.concluir_etapa("Download Dados", {'sucesso': len(data_d), 'falhas': len(falhas)})

def resample_tf(df, freq, min_days=4, min_days_monthly=10):
    if df is None or df.empty: return None
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex): df.index = pd.to_datetime(df.index)
    agg = {'Open':'first', 'High':'max', 'Low':'min', 'Close':'last', 'Volume':'sum'}
    df_r = df.resample(freq, closed='left', label='left').agg(agg)
    if freq.startswith('W'):
        counts = df.resample(freq, closed='left', label='left').count()['Close']
        df_r = df_r[counts >= min_days]
    elif freq in ('ME', 'M'):
        counts = df.resample(freq, closed='left', label='left').count()['Close']
        df_r = df_r[counts >= min_days_monthly]
    return df_r.dropna()

logger.iniciar_etapa("Resample Semanal/Mensal")
data_w, data_m = {}, {}
for t in tickers_liquidos:
    try:
        if t in data_d and not data_d[t].empty:
            df_d = data_d[t].copy()
            if isinstance(df_d.columns, pd.MultiIndex): df_d.columns = df_d.columns.droplevel(1)
            data_w[t] = resample_tf(df_d, 'W-FRI')
            data_m[t] = resample_tf(df_d, 'ME', min_days_monthly=10)
    except: continue
logger.concluir_etapa("Resample", {'semanais': len(data_w), 'mensais': len(data_m)})

nh_nl = calcular_nh_nl_simplificado(tickers_liquidos, data_w)
lad = calcular_lad(tickers_liquidos, data_d)
logger.log(f"📊 NH‑NL: {nh_nl}% | LAD: {lad['saldo'] if lad else 'N/A'}", "INFO")

logger.iniciar_etapa("Regime de Volatilidade")
ibov = None
for simbolo in ["^IBOV", "^BVSP", "BOVA11.SA"]:
    try:
        ibov_raw = yf.download(simbolo, period='3mo', interval='1d', progress=False)
        if not ibov_raw.empty and 'Close' in ibov_raw.columns:
            ibov = ibov_raw['Close'].dropna()
            if isinstance(ibov, pd.DataFrame): ibov = ibov.squeeze()
            if len(ibov) >= 60:
                logger.log(f"✅ IBOV via {simbolo}", "INFO")
                break
    except: pass
if ibov is not None and len(ibov) >= 60:
    regime_vol = detectar_regime_volatilidade(ibov)
    PARAMS_ATIVOS.clear()
    PARAMS_ATIVOS.update(PARAMS_ALTA_VOL if regime_vol == 'ALTA' else PARAMS_BAIXA_VOL)
else:
    regime_vol = 'BAIXA'
    PARAMS_ATIVOS.clear()
    PARAMS_ATIVOS.update(PARAMS_BAIXA_VOL)
logger.concluir_etapa("Regime", {'regime': regime_vol})

kelly_pct = fractional_kelly(WIN_RATE_ESTIMADO, PAYOFF_ESTIMADO, PARAMS_ATIVOS['kelly_frac'])
risco_maximo = CAPITAL_TOTAL * kelly_pct

def verificar_circuit_breakers():
    if not os.path.exists(ARQUIVO_LOG): return True, None
    try:
        with open(ARQUIVO_LOG, 'r', encoding='utf-8') as f: logs = json.load(f)
        hoje = datetime.now().date()
        trades = [l for l in logs if l['tipo'] == 'TRADE_FECHADO' and datetime.fromisoformat(l['timestamp']).date() == hoje]
        if not trades: return True, None
        pnl = sum(t['dados'].get('pnl_real', 0) for t in trades)
        if abs(pnl) / CAPITAL_TOTAL >= DRAWDOWN_MAX_DIARIO: return False, f"Drawdown >= {DRAWDOWN_MAX_DIARIO*100}%"
        perdas = 0
        for t in reversed(trades):
            if t['dados'].get('pnl_real', 0) < 0: perdas += 1
            else: break
        if perdas >= MAX_PERDAS_CONSECUTIVAS: return False, f"{perdas} perdas seguidas"
        return True, None
    except: return True, None

pode, motivo = verificar_circuit_breakers()
if not pode:
    logger.log(f"🛑 CIRCUIT BREAKER: {motivo}", "ALERT")
    raise SystemExit

oportunidades_swing, oportunidades_position = [], []
status_ativos = []  # Lista para guardar o resultado de cada ticker

def get_df(data, ticker):
    if data and isinstance(data, dict) and ticker in data:
        try:
            df = data[ticker].copy()
            if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.droplevel(1)
            df.rename(columns={'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}, inplace=True)
            return df
        except: pass
    return None

stats_filtros = {'total_analisados': 0, 'setup_aprovado_swing': 0, 'setup_aprovado_position': 0,
                 'bloqueios_preco': 0, 'bloqueios_risco': 0, 'bloqueios_confluencia': 0,
                 'bloqueios_mm200': 0, 'bloqueios_volume': 0, 'bloqueios_lta': 0,
                 'bloqueios_dow': 0, 'bloqueios_elliott': 0, 'bloqueios_fib': 0,
                 'bloqueios_ret': 0, 'bloqueios_stoch': 0, 'bloqueios_ma': 0,
                 'bloqueios_zona': 0, 'bloqueios_gatilho': 0, 'bloqueios_payoff': 0,
                 'bloqueios_corda': 0, 'bloqueios_alta_conf': 0, 'bloqueios_bollinger': 0,
                 'bloqueios_zona_wyckoff': 0,
                 'padroes_detectados': 0}

preco_minimo = PARAMS_ATIVOS.get('preco_minimo', PRECO_MINIMO)
risco_max_pct = PARAMS_ATIVOS.get('risco_percentual_maximo', RISCO_PERCENTUAL_MAXIMO)
gap_max_pct = PARAMS_ATIVOS['gap_max_pct']

logger.iniciar_etapa("Análise de Setups")

for i, ticker in enumerate(tickers_liquidos):
    if LOG_FILTROS_DETALHADO and i % 20 == 0: logger.log(f"Progresso: {i+1}/{len(tickers_liquidos)}", "DEBUG")
    df_w = get_df(data_w, ticker)
    df_d_local = get_df(data_d, ticker)
    df_m_local = get_df(data_m, ticker)
    if df_w is None or df_w.empty:
        status_ativos.append({'Ticker': ticker, 'Status': 'Sem dados', 'Filtro': 'dados_insuficientes'})
        continue
    stats_filtros['total_analisados'] += 1
    vol_fin = None
    try:
        vfc = (df_w['Volume'] * df_w['Close']).rolling(20).mean()
        vol_fin = vfc.iloc[-1] if pd.notna(vfc.iloc[-1]) else None
    except: pass

    motivo_recusa = None
    aprovado = False

    # ================= SWING TRADE (MOEDOR DE CARNE v8.4) =================
    res_swing = analisar_swing_trade(ticker, df_w=df_w, df_d=df_d_local)
    if res_swing:
        for r in res_swing:
            e = r['Entrada']
            rp = r['Risco (R$)'] / e

            # Pré-filtros
            if e < preco_minimo:
                motivo_recusa = motivo_recusa or 'Preço < mínimo'
                stats_filtros['bloqueios_preco'] += 1
                continue
            if rp < RISCO_PERCENTUAL_MINIMO or rp > risco_max_pct:
                motivo_recusa = motivo_recusa or 'Risco fora do intervalo'
                stats_filtros['bloqueios_risco'] += 1
                continue
            if r['Regime'] not in [1, 2] or r['Eficiência'] is None or r['Eficiência'] < 0.6:
                motivo_recusa = motivo_recusa or 'Confluência insuficiente'
                stats_filtros['bloqueios_confluencia'] += 1
                continue
            mm200w = df_w['Close'].rolling(200).mean().iloc[-1]
            if pd.notna(mm200w) and e < mm200w:
                motivo_recusa = motivo_recusa or 'Abaixo da MM200'
                stats_filtros['bloqueios_mm200'] += 1
                continue
            if PARAMS_ATIVOS.get('exigir_volume_anormal', False) and not r.get('Volume Anormal', True):
                motivo_recusa = motivo_recusa or 'Volume normal'
                stats_filtros['bloqueios_volume'] += 1
                continue
            # Validação da LTA
            res_lta = calcular_lta_adaptativo(df_w)
            if res_lta is None:
                motivo_recusa = motivo_recusa or 'LTA não definida'
                stats_filtros['bloqueios_lta'] += 1
                continue

            # Contexto de Armadilha
            alargamento_detectado = detectar_alargamento(df_w)
            is_arm = False
            if r['Direcao'] == 'COMPRA':
                lta_ref = r.get('Suporte', e)
                is_arm, _ = detectar_armadilha_lta(df_w, lta_ref, BANDA_ZONA_PCT)
            contexto_trap = is_arm and (alargamento_detectado is not None)

            # Guardião 1 – Dow
            dow = detectar_estrutura_dow(df_w)
            dow_ok = contexto_trap or (dow and dow['tendencia_dow'] == 'ALTA')
            if not dow_ok:
                motivo_recusa = 'Dow'
                stats_filtros['bloqueios_dow'] += 1
                continue

            # Guardião 2 – Elliott
            swing_highs, swing_lows = [], []
            for j in range(5, len(df_w)-5):
                if df_w['High'].values[j] >= max(df_w['High'].values[j-5:j+6]): swing_highs.append(df_w['High'].values[j])
                if df_w['Low'].values[j] <= min(df_w['Low'].values[j-5:j+6]): swing_lows.append(df_w['Low'].values[j])
            elliott_valido, _ = validar_elliott(df_w, swing_lows, swing_highs)
            elliott_ok = contexto_trap or elliott_valido
            if not elliott_ok:
                motivo_recusa = 'Elliott'
                stats_filtros['bloqueios_elliott'] += 1
                continue

            # Guardião 3 – Fibonacci
            fib_ret = calcular_fibonacci_retracao(df_w)
            fib_ok = True
            if fib_ret:
                fib_ok = (fib_ret['61.8%'] <= e <= fib_ret['38.2%'])
            if not fib_ok:
                motivo_recusa = 'Fibonacci'
                stats_filtros['bloqueios_fib'] += 1
                continue

            # Guardião 4 – Retângulo
            ret = detectar_retangulo(df_w)
            ret_ok = True
            if ret and 'suporte' in ret:
                ret_ok = e <= ret['suporte'] * 1.05
            if not ret_ok:
                motivo_recusa = 'Retângulo'
                stats_filtros['bloqueios_ret'] += 1
                continue

            # Guardião 5 – Estocástico
            stoch_k, _ = calcular_estocastico(df_w)
            stoch_ok = stoch_k is not None and stoch_k < 30
            if not stoch_ok:
                motivo_recusa = 'Estocástico'
                stats_filtros['bloqueios_stoch'] += 1
                continue

            # Guardião 6 – Médias
            mm200w_ant = df_w['Close'].rolling(200).mean().iloc[-5] if len(df_w) >= 200 else mm200w
            ma_ok = pd.notna(mm200w) and e > mm200w and mm200w > mm200w_ant
            if not ma_ok:
                motivo_recusa = 'Médias'
                stats_filtros['bloqueios_ma'] += 1
                continue

            # Guardião 7 – Toque na Zona (WYCKOFF v8.4)
            # Usa a nova função que valida com FECHO e volume adaptativo
            toca_zona, regime_wyckoff = validar_toque_zona_wyckoff(df_w, res_lta[0], BANDA_ZONA_PCT)
            if not toca_zona:
                motivo_recusa = 'Zona Wyckoff'
                stats_filtros['bloqueios_zona_wyckoff'] += 1
                continue

            # Guardião 8 – Gatilho
            pc = analisar_candle(df_w.iloc[-1], df_w.iloc[-2] if len(df_w) >= 2 else None)
            gatilho_ok = pc.get('martelo') or pc.get('engolfo_alta') or pc.get('kicker_alta') or pc.get('harami_alta')
            if not gatilho_ok:
                motivo_recusa = 'Gatilho'
                stats_filtros['bloqueios_gatilho'] += 1
                continue

            # Guardião 9 – Payoff 3:1
            p_real = calcular_payoff_real(e, r['Alvo 3:1'], r['Stop Loss'], PARAMS_ATIVOS['custos_pct'])
            payoff_ok = p_real >= 3.0
            if not payoff_ok:
                motivo_recusa = 'Payoff'
                stats_filtros['bloqueios_payoff'] += 1
                continue

            # Guardião 10 – Corda
            mm200w_val = df_w['Close'].rolling(200).mean().iloc[-1]
            dist_corda = (e - mm200w_val) / mm200w_val * 100 if pd.notna(mm200w_val) else 0
            corda_ok = dist_corda <= DIST_CORDA_MAX
            if not corda_ok:
                motivo_recusa = 'Corda'
                stats_filtros['bloqueios_corda'] += 1
                continue

            # Guardião 11 – Alta Confiabilidade
            if ALTA_CONFIABILIDADE:
                dados_padroes = {}
                oco_det = detectar_oco(df_w)
                if oco_det: dados_padroes[oco_det['tipo']] = True
                triplo = detectar_topo_fundo_triplo(df_w)
                if triplo: dados_padroes[triplo['tipo']] = True
                if ret: dados_padroes['Retangulo'] = True
                if alargamento_detectado: dados_padroes['Alargamento'] = True
                band = detectar_bandeira(df_w)
                flam = detectar_flamula(df_w)
                if band: dados_padroes['Bandeira'] = True
                if flam: dados_padroes['Flamula'] = True
                tri_asc = detectar_triangulo_ascendente(df_w)
                tri_desc = detectar_triangulo_descendente(df_w)
                if tri_asc: dados_padroes['Triangulo_asc'] = True
                if tri_desc: dados_padroes['Triangulo_desc'] = True
                ilha = detectar_ilha_reversao(df_w)
                if ilha: dados_padroes[ilha['tipo']] = True
                contagem = contar_padroes(dados_padroes)
                if contagem < 2:
                    motivo_recusa = 'Alta Confiança'
                    stats_filtros['bloqueios_alta_conf'] += 1
                    continue

            # Guardião 12 – Bollinger
            if USAR_GATILHO_BOLLINGER:
                bollinger_ok = detectar_gatilho_bollinger(df_w)
                if not bollinger_ok:
                    motivo_recusa = 'Bollinger'
                    stats_filtros['bloqueios_bollinger'] += 1
                    continue

            # Se chegou aqui, passou em todos os guardiões
            aprovado = True
            fat_liq = min(1.0, vol_fin / LIMITE_LIQUIDEZ_FINANCEIRA) if pd.notna(vol_fin) else 0.5
            lote_base = int(risco_maximo / r['Risco (R$)'])
            lote_aj = int(lote_base * fat_liq)
            if lote_aj == 0:
                motivo_recusa = 'Lote zero'
                aprovado = False
                continue

            # Padrões detectados
            padroes = []
            for nome, val in pc.items():
                if val: padroes.append(nome)
            if oco_det: padroes.append(oco_det['tipo'])
            if tri_asc: padroes.append('Triang_asc')
            if padroes: stats_filtros['padroes_detectados'] += len(padroes)

            alvo_final = r['Alvo 3:1']
            if oco_det and 'alvo' in oco_det:
                if oco_det['alvo'] > alvo_final:
                    alvo_final = oco_det['alvo']

            r.update({'Alvo Recomendado': alvo_final, 'Método Alvo': '3:1', 'Payoff Real': p_real,
                       'Lote': lote_aj, 'Padrões Detectados': ', '.join(padroes) if padroes else 'Nenhum',
                       'Filtro Bloqueador': '✓ Aprovado (Wyckoff)', 'Regime Wyckoff': regime_wyckoff})
            stats_filtros['setup_aprovado_swing'] += 1
            oportunidades_swing.append(r)
            break  # Apenas um setup por ticker

    # Regista o status final do ticker
    if aprovado:
        status_ativos.append({'Ticker': ticker, 'Status': '✅ APROVADO', 'Filtro': 'Nenhum'})
    else:
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': motivo_recusa or 'Sem setup'})

logger.concluir_etapa("Análise de Setups", stats_filtros)

# ============================
# LOG DETALHADO POR TICKER
# ============================
if LOG_DETALHADO_TICKER and status_ativos:
    logger.log("\n📋 RESUMO POR TICKER:", "INFO")
    df_status = pd.DataFrame(status_ativos)
    aprovados = df_status[df_status['Status'] == '✅ APROVADO']
    recusados = df_status[df_status['Status'] == '❌ Recusado']
    logger.log(f"   Aprovados: {len(aprovados)} | Recusados: {len(recusados)}", "INFO")
    if len(recusados) > 0:
        logger.log("\n   ❌ Recusados (motivo):", "INFO")
        for _, row in recusados.iterrows():
            logger.log(f"      {row['Ticker']} → {row['Filtro']}", "INFO")
    if len(aprovados) > 0:
        logger.log("\n   ✅ Aprovados:", "INFO")
        for _, row in aprovados.iterrows():
            logger.log(f"      {row['Ticker']}", "INFO")
    csv_status = f"status_ativos_{datetime.now().strftime('%Y%m%d')}.csv"
    df_status.to_csv(csv_status, index=False)
    logger.log(f"\n📁 Status dos ativos guardado em '{csv_status}'", "INFO")

if len(oportunidades_swing) > MAX_SETUPS_POR_DIA:
    oportunidades_swing = sorted(oportunidades_swing, key=lambda x: x['Payoff Real'], reverse=True)[:MAX_SETUPS_POR_DIA]
if len(oportunidades_position) > MAX_SETUPS_POR_DIA:
    oportunidades_position = sorted(oportunidades_position, key=lambda x: x['Payoff Real'], reverse=True)[:MAX_SETUPS_POR_DIA]

logger.log(f"\n🎯 Swing: {len(oportunidades_swing)} | Position: {len(oportunidades_position)} setups", "RESULTADO")
logger.log(f"   Kelly: {kelly_pct*100:.2f}% | Regime: {regime_vol} | NH‑NL: {nh_nl}% | LAD: {lad['saldo'] if lad else 'N/A'}", "RESULTADO")
logger.log(f"   🛡️ Bloqueios – Preço:{stats_filtros['bloqueios_preco']} Risco:{stats_filtros['bloqueios_risco']} Confl:{stats_filtros['bloqueios_confluencia']} MM200:{stats_filtros['bloqueios_mm200']} Vol:{stats_filtros['bloqueios_volume']} LTA:{stats_filtros['bloqueios_lta']}", "RESULTADO")
logger.log(f"   🛡️ Guardiões – Dow:{stats_filtros['bloqueios_dow']} Elliott:{stats_filtros['bloqueios_elliott']} Fib:{stats_filtros['bloqueios_fib']} Ret:{stats_filtros['bloqueios_ret']} Stoch:{stats_filtros['bloqueios_stoch']} MA:{stats_filtros['bloqueios_ma']} ZonaW:{stats_filtros['bloqueios_zona_wyckoff']} Gat:{stats_filtros['bloqueios_gatilho']} Pay:{stats_filtros['bloqueios_payoff']} Corda:{stats_filtros['bloqueios_corda']} AltaC:{stats_filtros['bloqueios_alta_conf']} BB:{stats_filtros['bloqueios_bollinger']}", "RESULTADO")

if oportunidades_swing: enviar_email_ou_exibir(oportunidades_swing, "Swing Trade", regime_vol, nh_nl, lad)
if oportunidades_position: enviar_email_ou_exibir(oportunidades_position, "Position Trade", regime_vol, nh_nl, lad)

logger.resumo_final()
logger.log("✅ v8.4 Wyckoff Surgical concluído", "SUCCESS")
print("\n✅ Execução concluída.")